In [51]:
import os
import json
import urllib.request

# 1. Setup local raw data directory relative to the notebooks folder
raw_dir = "../data/raw"
os.makedirs(raw_dir, exist_ok=True)

# Define how many matches you want to download initially
MAX_MATCHES = 50 

print("Fetching competition data to locate available match IDs...")

# Target a major, high-volume league season (La Liga 2017/2018)
competitions_url = "https://raw.githubusercontent.com/statsbomb/open-data/master/data/matches/11/90.json"

try:
    # Read the competition match index file
    with urllib.request.urlopen(competitions_url) as response:
        match_data = json.loads(response.read().decode('utf-8'))
    
    # Extract unique match IDs
    match_ids = [str(match['match_id']) for match in match_data][:MAX_MATCHES]
    print(f"Discovered {len(match_ids)} matches available for downloading.")

    # 2. Iterative Ingestion Loop
    downloaded_count = 0
    for i, match_id in enumerate(match_ids):
        destination_path = os.path.join(raw_dir, f"{match_id}.json")
        
        # Guard clause: avoid re-downloading files that exist locally
        if os.path.exists(destination_path):
            continue
            
        event_url = f"https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/{match_id}.json"
        
        try:
            urllib.request.urlretrieve(event_url, destination_path)
            downloaded_count += 1
            if downloaded_count % 10 == 0 or i == len(match_ids) - 1:
                print(f"Progress: Checked {i+1}/{len(match_ids)} files... Downloaded {downloaded_count} new matches.")
        except Exception as file_error:
            print(f"Skipping match {match_id} due to an error: {file_error}")
            
    print(f"\n✅ Data collection complete! Local files are saved in: {raw_dir}")

except Exception as global_error:
    print(f"Could not connect to StatsBomb index repository: {global_error}")

Fetching competition data to locate available match IDs...
Discovered 35 matches available for downloading.

✅ Data collection complete! Local files are saved in: ../data/raw


In [52]:
import json
import pandas as pd
import numpy as np

test_match_file = "../data/raw/3764440.json"

#Convert the json file to a pandas dataframe
with open(test_match_file, 'r', encoding='utf-8') as f:
    match_data = json.load(f)

df = pd.DataFrame(match_data)

print(f"Total number of match events logged: {len(df)}")

Total number of match events logged: 4160


In [53]:
df.columns

Index(['id', 'index', 'period', 'timestamp', 'minute', 'second', 'type',
       'possession', 'possession_team', 'play_pattern', 'team', 'duration',
       'tactics', 'related_events', 'player', 'position', 'location', 'pass',
       'carry', 'under_pressure', 'dribble', 'shot', 'goalkeeper', 'out',
       'ball_recovery', 'duel', 'ball_receipt', 'clearance', 'off_camera',
       'counterpress', 'interception', 'foul_won', 'foul_committed',
       'substitution', '50_50', 'injury_stoppage'],
      dtype='str')

In [54]:
df.iloc[101]

id                                                         7971a576-07bb-41c2-8e43-1e51dabfc2b3
index                                                                                       102
period                                                                                        1
timestamp                                                                          00:02:37.544
minute                                                                                        2
second                                                                                       37
type                                                                {'id': 43, 'name': 'Carry'}
possession                                                                                    5
possession_team                                                {'id': 217, 'name': 'Barcelona'}
play_pattern                                                  {'id': 1, 'name': 'Regular Play'}
team                                    

In [55]:
df['event_type'] = df['type'].apply(lambda x: x.get('name') if isinstance (x,dict) else None)
df['event_type'].tail()

4155             Pass
4156    Ball Receipt*
4157     Interception
4158         Half End
4159         Half End
Name: event_type, dtype: str

In [56]:
df_pass = df[df['event_type'] == 'Pass'].copy()
df_pass['event_type'].head(10)

4     Pass
7     Pass
10    Pass
13    Pass
16    Pass
21    Pass
24    Pass
27    Pass
30    Pass
33    Pass
Name: event_type, dtype: str

In [57]:
pd.set_option('display.max_colwidth', None)
print(df_pass['pass'].head(50))

4                                                                         {'recipient': {'id': 24517, 'name': 'José Raúl Gutiérrez Parejo'}, 'length': 15.597436, 'angle': 2.428794, 'height': {'id': 1, 'name': 'Ground Pass'}, 'end_location': [49.2, 50.3], 'body_part': {'id': 38, 'name': 'Left Foot'}, 'type': {'id': 65, 'name': 'Kick Off'}}
7                                                                                                                  {'recipient': {'id': 24169, 'name': 'Gonzalo Cacicedo Verdú'}, 'length': 17.520845, 'angle': -2.8342814, 'height': {'id': 1, 'name': 'Ground Pass'}, 'end_location': [33.4, 46.1], 'body_part': {'id': 40, 'name': 'Right Foot'}}
10                                                                                                             {'recipient': {'id': 9857, 'name': 'José Manuel Sánchez Guillén'}, 'length': 27.011848, 'angle': -1.5411754, 'height': {'id': 1, 'name': 'Ground Pass'}, 'end_location': [35.1, 18.3], 'body_part': {'id': 40, 

In [58]:
#I see that when the pass is Complete there is not a outcome written, and when it's incomplete there is
df_pass['outcome'] = df_pass['pass'].apply(lambda x: x.get('outcome') if isinstance (x, dict) else None)
df_pass['outcome'] = df_pass['outcome'].fillna('Complete')
df_pass['outcome'].head()

4     Complete
7     Complete
10    Complete
13    Complete
16    Complete
Name: outcome, dtype: object

In [59]:
df_pass['outcome'] = (df_pass['outcome'] == 'Complete').astype(int)
df_pass['outcome'].head()

4     1
7     1
10    1
13    1
16    1
Name: outcome, dtype: int64

In [60]:
df_pass['outcome'].value_counts(normalize=True)

outcome
1    0.891358
0    0.108642
Name: proportion, dtype: float64

In [61]:
df_pass['under_pressure']

4        NaN
7        NaN
10       NaN
13       NaN
16       NaN
        ... 
4134    True
4137    True
4141    True
4152     NaN
4155     NaN
Name: under_pressure, Length: 1215, dtype: object

In [62]:
#I see that the column is NaN when it's not under pressure and "True" when it is
#I will now make the column binary

df_pass['under_pressure'] = (df_pass['under_pressure']).fillna(False).astype(int)

In [63]:
df_pass['under_pressure']

4       0
7       0
10      0
13      0
16      0
       ..
4134    1
4137    1
4141    1
4152    0
4155    0
Name: under_pressure, Length: 1215, dtype: int64

In [80]:
df_pass['under_pressure'].value_counts(normalize=True)

under_pressure
0    0.898765
1    0.101235
Name: proportion, dtype: float64

In [64]:
df_pass.iloc[0]

id                                                                                                                                                                                                                                                       23743e50-fbe4-4929-938e-9b3ab39ff4ff
index                                                                                                                                                                                                                                                                                       5
period                                                                                                                                                                                                                                                                                      1
timestamp                                                                                                                                     

In [65]:
#I will flatten the data that have a simple dictionary
columns_with_name = ['type', 'possession_team', 'play_pattern', 'team', 'player', 'position']

for col in columns_with_name:
    df_pass[col] = df_pass[col].apply(lambda x: x.get('name') if isinstance (x, dict) else None)

df_pass.iloc[0]

id                                                                                                                                                                                                                                                       23743e50-fbe4-4929-938e-9b3ab39ff4ff
index                                                                                                                                                                                                                                                                                       5
period                                                                                                                                                                                                                                                                                      1
timestamp                                                                                                                                     

In [66]:
df_pass['pass'].iloc[0]

{'recipient': {'id': 24517, 'name': 'José Raúl Gutiérrez Parejo'},
 'length': 15.597436,
 'angle': 2.428794,
 'height': {'id': 1, 'name': 'Ground Pass'},
 'end_location': [49.2, 50.3],
 'body_part': {'id': 38, 'name': 'Left Foot'},
 'type': {'id': 65, 'name': 'Kick Off'}}

In [67]:
#I will now flatten the pass column
#Using the pd.json_normalize() I can do this easier without having to do .apply for everything inisde the column 'pass'
#I tried the above but took way to much time to run, so I will just do the .apply method to every key I need
#I will use functions for my convenience

def apply_simple(df, colname, key):
    return df[colname].apply(lambda x: x.get(key) if isinstance(x,dict) else None)

def apply_complex(df, colname, key1, key2):
    return df[colname].apply(lambda x: x.get(key1, {}).get(key2) if isinstance (x,dict) else None)

In [68]:
df_pass['recipient'] = apply_complex(df_pass, 'pass', 'recipient', 'name')
df_pass['length'] = apply_simple(df_pass, 'pass', 'length')
df_pass['angle'] = apply_simple(df_pass, 'pass', 'angle')
df_pass['height'] = apply_complex(df_pass, 'pass', 'height', 'name')
df_pass['end_location'] = apply_simple(df_pass, 'pass', 'end_location')
df_pass['body_part'] = apply_complex(df_pass, 'pass', 'body_part', 'name')
df_pass['type'] = apply_complex(df_pass, 'pass', 'type', 'name')

In [69]:
df_pass.iloc[0]

id                                                                                                                                                                                                                                                       23743e50-fbe4-4929-938e-9b3ab39ff4ff
index                                                                                                                                                                                                                                                                                       5
period                                                                                                                                                                                                                                                                                      1
timestamp                                                                                                                                     

In [70]:
#I will now get the coordinates for starting and ending location of the pass
df_pass['start_x'] = df_pass['location'].apply(lambda x: x[0] if isinstance(x, list) else None)
df_pass['start_y'] = df_pass['location'].apply(lambda x: x[1] if isinstance(x, list) else None)
df_pass['end_x'] = df_pass['end_location'].apply(lambda x: x[0] if isinstance(x, list) else None)
df_pass['end_y'] = df_pass['end_location'].apply(lambda x: x[1] if isinstance(x, list) else None)

In [71]:
df_pass[['start_x', 'start_y', 'end_x', 'end_y']].head()

,start_x,start_y,end_x,end_y
4,61.0,40.1,49.2,50.3
7,50.1,51.4,33.4,46.1
10,34.3,45.3,35.1,18.3
13,34.5,22.8,26.5,55.5
16,34.1,63.3,89.6,56.1


In [72]:
#Create the function that calculates the angle between two points
def two_point_angle(x1, y1, x2, y2):
    return np.arctan2(y2-y1, x2-x1)

In [73]:
df_pass['angle_calc'] = two_point_angle(df_pass['start_x'], df_pass['start_y'], df_pass['end_x'], df_pass['end_y'])

In [74]:
#Compare between the calculated and given values of the angle
(df_pass['angle_calc'] - df_pass['angle']).describe()

count    1.215000e+03
mean     1.213277e-10
std      5.003024e-08
min     -2.061205e-07
25%     -2.575655e-08
50%      0.000000e+00
75%      2.682618e-08
max      1.940907e-07
dtype: float64

In [75]:
df_pass['angle'] = df_pass['angle_calc']
df_pass.drop(columns=['angle_calc'], inplace=True)

In [76]:
#Create a function for the Pythagorean theorem
def Pythagorean(x1, y1, x2, y2):
    return np.sqrt((x2 - x1)**2 + (y2 - y1)**2)

In [77]:
#Calculate the length of the pass, the distance of the receiver from the goal, and the distance of the receiver from the central line of the pitch
df_pass['length_calc'] = Pythagorean(df_pass['start_x'], df_pass['start_y'], df_pass['end_x'], df_pass['end_y'])
opponentgoal_x = 120
opponentgoal_y = 40
df_pass['dist_receiv_goal'] = Pythagorean(opponentgoal_x, opponentgoal_y, df_pass['end_x'], df_pass['end_y'])
df_pass['dist_receiv_middle'] = df_pass['end_x'] - 60

In [78]:
#Compare the calculated and given values of the length of the pass
(df_pass['length_calc'] - df_pass['length']).describe()

count    1.215000e+03
mean     3.108854e-08
std      6.646040e-07
min     -3.340939e-06
25%     -2.813428e-07
50%      0.000000e+00
75%      3.446706e-07
max      5.793543e-06
dtype: float64

In [79]:
#Drop the given value of the pass
df_pass['length'] = df_pass['length_calc']
df_pass.drop(columns = ['length_calc'], inplace=True)